In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Importing dataset

In [19]:
data = pd.read_csv('../M1_final.csv')
data.head()

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,TAIL_NUM,DEST,DEP_DELAY,CRS_ELAPSED_TIME,DISTANCE,CRS_DEP_M,...,Dew Point,Humidity,Wind,Wind Speed,Wind Gust,Pressure,Condition,sch_dep,sch_arr,TAXI_OUT
0,11,1,5,B6,N828JB,CHS,-1,124,636,324,...,34,58,W,25,38,29.86,Fair / Windy,9,17,14
1,11,1,5,B6,N992JB,LAX,-7,371,2475,340,...,34,58,W,25,38,29.86,Fair / Windy,9,17,15
2,11,1,5,B6,N959JB,FLL,40,181,1069,301,...,34,58,W,25,38,29.86,Fair / Windy,9,17,22
3,11,1,5,B6,N999JQ,MCO,-2,168,944,345,...,34,58,W,25,38,29.86,Fair / Windy,9,17,12
4,11,1,5,DL,N880DN,ATL,-4,139,760,360,...,32,58,W,24,35,29.91,Fair / Windy,9,17,13


In [20]:
# Drop tail num
data.drop(columns=['TAIL_NUM'], inplace=True)

In [21]:
# Create target variable
data['is_delayed'] = np.where(
    data['DEP_DELAY'] >= 15,
    1,
    0
)
data.sample(5)

# Drop dep delay column
data.drop(columns=['DEP_DELAY'], inplace=True)

In [22]:
# Drop missing rows
data.dropna(inplace=True)
data.shape

(28818, 22)

In [23]:
# Train test split

# Stratified Train test split
from sklearn.model_selection import StratifiedShuffleSplit

x = data.drop('is_delayed', axis=1)
y = data['is_delayed']

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in splitter.split(data, data['is_delayed']):
    x_train = x.iloc[train_index]
    x_test = x.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

In [24]:
# Transformers
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer

In [25]:
from sklearn.base import BaseEstimator, TransformerMixin

class WindDirectionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='Wind'):
        self.column = column
        self.wind_dict = {
            'NNW': 340, 'CALM': 0, 'NNE': 20, 'NE': 45, 'VAR': 0, 'WSW': 230, 
            'S': 180, 'SSW': 200, 'WNW': 290, 'ESE': 115, 'N': 360, 'SW': 225, 
            'E': 90, 'W': 270, 'SSE': 155, 'ENE': 70, 'NW': 315, 'SE': 135
        }

    def fit(self, X, y=None):
        return self
    

    def transform(self, X):
        X = X.copy()
        # Map wind directions to degrees
        X['wind_deg'] = X[self.column].map(self.wind_dict)
        # Convert to radians
        X['wind_rad'] = np.deg2rad(X['wind_deg'])
        #Compute sin and cos
        X['wind_sin'] = np.sin(X['wind_rad'])
        X['wind_cos'] = np.cos(X['wind_rad'])
        # Drop original columns
        X = X.drop(columns=[self.column, 'wind_deg', 'wind_rad'])
        return X

In [26]:
class DewPointTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='Dew Point'):
        self.column = column
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()

        # Clean column
        X[self.column] = (
            X[self.column].astype(str).str.replace('\xa0', '', regex=False).str.strip()
        )
        # Convert them into numeric values
        X[self.column] = pd.to_numeric(X[self.column], errors='coerce')

        return X

### Pipeline

In [27]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28818 entries, 0 to 28819
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MONTH              28818 non-null  int64  
 1   DAY_OF_MONTH       28818 non-null  int64  
 2   DAY_OF_WEEK        28818 non-null  int64  
 3   OP_UNIQUE_CARRIER  28818 non-null  object 
 4   DEST               28818 non-null  object 
 5   CRS_ELAPSED_TIME   28818 non-null  int64  
 6   DISTANCE           28818 non-null  int64  
 7   CRS_DEP_M          28818 non-null  int64  
 8   DEP_TIME_M         28818 non-null  int64  
 9   CRS_ARR_M          28818 non-null  int64  
 10  Temperature        28818 non-null  int64  
 11  Dew Point          28818 non-null  object 
 12  Humidity           28818 non-null  int64  
 13  Wind               28818 non-null  object 
 14  Wind Speed         28818 non-null  int64  
 15  Wind Gust          28818 non-null  int64  
 16  Pressure           28818 no

In [28]:
from sklearn.preprocessing import OrdinalEncoder

preprocessor = ColumnTransformer(transformers=[
    ('Wind transformer', WindDirectionTransformer(column='Wind'), ['Wind']),
    ('Dew Point Transformer', DewPointTransformer(column='Dew Point'), ['Dew Point']),
    ('Categorical', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ['DEST', 'OP_UNIQUE_CARRIER', 'Condition'])
], remainder='passthrough')

In [29]:
from sklearn.tree import DecisionTreeClassifier

decisionTree = DecisionTreeClassifier(
    max_depth=5,             # Don't let the tree grow too deep
    min_samples_leaf=20,     # A final decision (leaf) must be based on at least 20 flights
    random_state=42
)

In [30]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('PreProcessing', preprocessor),
    ('Train Decision Tree', decisionTree)
])

In [31]:
pipe.fit(x_train, y_train)

,steps,"[('PreProcessing', ...), ('Train Decision Tree', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Wind transformer', ...), ('Dew Point Transformer', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [32]:
y_pred = pipe.predict(x_test)
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(5764,))

In [33]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.8860166551006246

### Cross validation

In [34]:
from sklearn.model_selection import cross_val_score

cross_val_score(pipe, data.drop(columns=['is_delayed']), data['is_delayed'], cv=10, scoring='accuracy').mean()

np.float64(0.8148773184575002)

In [35]:
from sklearn.model_selection import GridSearchCV

# 1. Define the hyperparameter grid to search
# The keys of the dictionary must match the names of the steps in your pipeline,
# followed by a double underscore and the hyperparameter name.
# e.g., 'estimator_name__parameter_name'

param_grid = {
    'Train Decision Tree__criterion': ['gini', 'entropy'],
    'Train Decision Tree__max_depth': [5, 7, 10, 15],
    'Train Decision Tree__min_samples_leaf': [15, 20, 25],
    'Train Decision Tree__min_samples_split': [20, 40, 60],
    'Train Decision Tree__max_features': [None, 'sqrt', 'log2'],
    'Train Decision Tree__class_weight': [None, 'balanced']
}

In [36]:
# 2. Instantiate GridSearchCV
# We will use the existing pipeline 'pipe'
# cv=5 means we will use 5-fold cross-validation for evaluation
# scoring='accuracy' is the metric we want to optimize
# n_jobs=-1 will use all available CPU cores to speed up the process

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy', # You could also try 'f1', 'precision', or 'recall' for imbalanced data
    n_jobs=-1,
    verbose=2
)

In [37]:
# 3. Fit the grid search to the training data
# This will start the search process and can take some time
print("Starting grid search... this may take a while.")
grid_search.fit(x_train, y_train)

Starting grid search... this may take a while.
Fitting 5 folds for each of 432 candidates, totalling 2160 fits
[CV] END Train Decision Tree__class_weight=None, Train Decision Tree__criterion=gini, Train Decision Tree__max_depth=5, Train Decision Tree__max_features=None, Train Decision Tree__min_samples_leaf=15, Train Decision Tree__min_samples_split=60; total time=   0.3s
[CV] END Train Decision Tree__class_weight=None, Train Decision Tree__criterion=gini, Train Decision Tree__max_depth=5, Train Decision Tree__max_features=None, Train Decision Tree__min_samples_leaf=15, Train Decision Tree__min_samples_split=20; total time=   0.3s
[CV] END Train Decision Tree__class_weight=None, Train Decision Tree__criterion=gini, Train Decision Tree__max_depth=5, Train Decision Tree__max_features=None, Train Decision Tree__min_samples_leaf=15, Train Decision Tree__min_samples_split=40; total time=   0.4s
[CV] END Train Decision Tree__class_weight=None, Train Decision Tree__criterion=gini, Train Decis

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'Train Decision Tree__class_weight': [None, 'balanced'], 'Train Decision Tree__criterion': ['gini', 'entropy'], 'Train Decision Tree__max_depth': [5, 7, ...], 'Train Decision Tree__max_features': [None, 'sqrt', ...], ...}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Wind transformer', ...), ('Dew Point Transformer', ...), ...]"


In [38]:
# 4. Print the best parameters and the best score
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(x_test)
print(f"Test set accuracy with best model: {accuracy_score(y_test, y_pred_best):.4f}")


Test set accuracy with best model: 0.9672
